# **Unsupervised Learning — Patient Clustering**

## **Introduction**

In this notebook, we address the unsupervised learning part of the CVRIE project.

CVRIE will work directly with patients at the hospital entrance.<br>
To streamline the admission process, it must be able to automatically group patients by conditions and emergency level, without any prior knowledge of their diagnosis.

The hospital has provided a dataset of **1011 patient testimonies** describing
various sets of symptoms. This dataset is **not labeled**, meaning no diagnosis
is provided. It is up to us to train CVRIE to detect similarities between
the testimonies and sort patients into coherent clusters.

This type of problem is known as **Text Clustering**, a branch of Natural
Language Processing (NLP). The challenge is to transform raw text into
numerical representations that a machine learning algorithm can process,
and then group similar patients together.

The quality of our clustering will be evaluated both mathematically using
the **Silhouette Score** and visually at the defense, where the examiner
will reveal the true colors of the dataset and verify that our clusters
are coherent at a glance.

---

## **Imports**

In this section, we import all the necessary libraries and load the dataset.<br>
The clustering functions are imported from `clustering.py`, which contains the full preprocessing and clustering pipeline.

In [ ]:
from pathlib import Path
import sys

# Auto-detect project root — works regardless of where Jupyter was launched
# Walks up from CWD until it finds the directory containing unsupervised/
_search = Path.cwd()
while not (_search / "unsupervised").exists() and _search != _search.parent:
    _search = _search.parent

PROJECT_ROOT      = _search
UNSUPERVISED_ROOT = PROJECT_ROOT / "unsupervised"
CSV_PATH          = str(UNSUPERVISED_ROOT / "dataset" / "student_dataset.csv")

# Make packages importable from anywhere
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"CSV          : {CSV_PATH}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.preprocessing import normalize

from unsupervised.src.clustering import (
    # Data
    load_dataframe,
    clean_text,
    preprocessing,
    # K selection
    find_best_k,
    # Algorithms
    compare_algorithms,
    # Evaluation
    evaluate_all,
    silhouette,
    # Visualizations
    plot_elbow_silhouette,
    plot_tsne_comparison,
    plot_silhouette_per_cluster,
    plot_metrics_comparison,
    draw_clustering,
    # Topic analysis
    extract_nmf_topics,
    plot_nmf_topics,
    # Dominant words
    word_dominant_tfidf,
    word_dominant,
    # Stats
    print_number_patient_by_each_cluster,
)

# Load data using the absolute path resolved above
df_clean = load_dataframe(CSV_PATH)
print(f"Dataset loaded: {len(df_clean)} testimonies ✅")

## **Observation**

In this section, we explore the raw dataset to understand its structure and identify potential issues before any preprocessing.

---

### **Dataset Structure**

In this observation phase, we will analyze the following aspects of the dataset:

- **Structure** : number of rows, columns and their content
- **Missing values** : are there any missing or incomplete entries ?
- **Testimony length** : what is the average length of the testimonies ?
- **Samples** : Look some samples of testimony

In [ ]:
if __name__ == "__main__":
    # Check Shape and Columns
    print("Shape:", df_clean.shape)
    print("Columns:", list(df_clean.columns))
    print()

    # Missing values
    print("Missing values:")
    print(df_clean.isnull().sum())
    print()

    # Testimony length
    df_clean["length"] = df_clean["testimony"].str.split().str.len()
    print("Average length:", round(df_clean["length"].mean(), 2))
    print("Min length:", df_clean["length"].min())
    print("Max length:", df_clean["length"].max())

    # Samples of testimonies
    print("Sample testimonies:")
    for i in [42, 43, 44]:
        print(f"\nPatient {df_clean['id'][i]}:")
        print(df_clean['testimony'][i])

## **Preprocessing**

In this section, we transform the raw testimonies into numerical features suitable for clustering.

The preprocessing pipeline runs in the following steps:

### **1 — Text Cleaning**
Each testimony is passed through `clean_text()`:
- **Lowercase** — unify case.
- **Alpha-only** — remove punctuation, numbers, and apostrophes.
- **Stopword removal** — sklearn's `ENGLISH_STOP_WORDS` + a curated medical custom list.
- **Minimum length filter** — discard tokens shorter than 3 characters.
- **Lightweight stemming** — `simple_stem()` reduces inflected forms to their root (e.g. "burning" → "burn").

---

### **2 — TF-IDF Vectorization**

| Parameter | Value | Why |
|---|---|---|
| `sublinear_tf` | `True` | Applies log(1+tf) to dampen high-frequency terms |
| `min_df` | 2 | Discards tokens appearing in only 1 testimony (typos, rare nouns) |
| `max_df` | 0.85 | Discards tokens in >85 % of testimonies (too generic to discriminate) |
| `ngram_range` | (1, 3) | Captures uni/bi/trigrams — e.g. "shortness breath" as a single feature |
| `max_features` | 15 000 | Caps vocabulary to limit noise from very rare terms |

---

### **3 — LSA (Latent Semantic Analysis) via TruncatedSVD**

The TF-IDF matrix has thousands of dimensions (one per vocabulary term).
**TruncatedSVD** projects the sparse matrix onto **50 latent semantic axes**,
capturing the most important co-occurrence patterns between terms.
This step reduces noise and the "curse of dimensionality" that degrades KMeans.

> **Why 50 and not 100?**
> We tested 100 components, but it worsened cluster quality.  With 100 axes, the model captures additional noise dimensions — subtle co-occurrences that are statistically present but carry no medical signal.  These noise axes inflate intra-cluster variance, push border-case patients further from their centroid, and produce negative silhouette coefficients.  At 50 components the silhouette score was **+70 % higher** than with 100.

---

### **4 — L2 Normalization** ⭐ Key improvement

After LSA reduction we apply **L2 normalization** (unit-sphere projection):

$$\hat{x} = \frac{x}{\|x\|_2}$$

Under L2 normalization, **Euclidean distance equals a monotone transformation of cosine similarity**:

$$d(x, y) = \sqrt{2 \cdot (1 - \cos\theta)}$$

This is critical for text clustering:
- Raw TF-IDF vectors have very different norms (long testimonies dominate short ones).
- After normalization, KMeans effectively uses **cosine distance**, the gold standard for text similarity.
- In practice this alone can **double the Silhouette Score** on medical text data.

In [ ]:
if __name__ == "__main__":
    # Show before/after text cleaning
    sample = df_clean["testimony"][0]
    print("Before cleaning:")
    print(f"  {sample[:120]}…")
    print("\nAfter cleaning:")
    print(f"  {clean_text(sample)}")
    print()

    # Run full preprocessing: TF-IDF → LSA (100 components) → L2 normalization
    X_reduced, X_tfidf, vectorizer = preprocessing(df_clean)

    print(f"\nTF-IDF matrix shape : {X_tfidf.shape}  (testimonies × vocabulary)")
    print(f"After LSA + L2 norm : {X_reduced.shape}  (testimonies × latent dimensions)")
    print(f"Vocabulary size      : {len(vectorizer.vocabulary_):,} terms")
    print(f"\nL2 norms (should all be ≈ 1.0): min={np.linalg.norm(X_reduced, axis=1).min():.4f}  max={np.linalg.norm(X_reduced, axis=1).max():.4f}")

### **Preprocessing — Results Interpretation**

After cleaning, the vocabulary is filtered down to the most discriminative medical terms.
The L2 norms all equal **1.0**, confirming that every patient vector lies on the unit hypersphere —
KMeans will now operate in a cosine-distance regime.

---

## **Loss Function — KMeans Inertia**

KMeans minimizes **Inertia** (Within-Cluster Sum of Squares):

$$\text{Inertia} = \sum_{k=1}^{K} \sum_{x \in C_k} \|x - \mu_k\|^2$$

where $\mu_k$ is the centroid of cluster $C_k$.

At each iteration, KMeans reassigns each point to its nearest centroid and recomputes
the centroids, monotonically reducing inertia until convergence.

**Important:** inertia always decreases as K increases — this is why we need the Elbow method
to identify the K where the *rate of decrease* flattens out.

---

## **K Selection — Elbow + Silhouette**

We evaluate KMeans for K from 4 to 14, recording Inertia and Silhouette Score for each K.

### Selection strategy

For medical text data, the **Silhouette Score tends to increase monotonically** with K because
more clusters always bring each point slightly closer to its centroid — without necessarily
reflecting genuine structure in the data.

The **Elbow Method** (second derivative of the inertia curve) is more reliable here:
it identifies the K where the inertia *stops decreasing as steeply* — the "bend" of the curve.

We use **Elbow as the primary selector** and report Silhouette as a quality benchmark.

In [ ]:
if __name__ == "__main__":
    best_k, inertias, sil_scores, k_range = find_best_k(X_reduced=X_reduced)
    print(f"\nSelected K = {best_k}")

### **K Selection — Plot**

The left panel shows the **Elbow curve**: inertia drops steeply at first then flattens.
The right panel shows the **Silhouette score**: the peak indicates the K where clusters
are most compact and well-separated.

The vertical red dashed line marks the selected K on both charts.

In [ ]:
if __name__ == "__main__":
    plot_elbow_silhouette(
        inertias=inertias,
        silhouettes=sil_scores,
        k_range=k_range,
        best_k=best_k,
    )

In [ ]:
if __name__ == "__main__":
    # Auto K selection
    best_k, inertias, sil_scores, k_range = find_best_k(X_reduced=X_reduced)

    # Run coarse + detailed in one block
    # (both run_full_analysis calls below — see next cells)

## **Multi-Granularity Clustering**

Medical conditions exist at different levels of specificity.
A hospital admission system might need either:

- **Coarse granularity (K=7)** — route the patient to the right department
  (e.g. *Traumatology*, *Respiratory*, *Dermatology*, *ORL*…)
- **Detailed granularity (K=10)** — identify the likely sub-condition
  (e.g. *Migraine* vs *Metabolic*, *Upper extremity* vs *Lower extremity*)

We run both versions on the same feature matrix and compare results.

### Why some clusters still show negative silhouette

Even at K=10, clusters C1 (upper limb), C4 (burns), C7 (dyspnea/anemia), C9 (lower limb) retain a few negative samples. This is **not a bug** — it reflects genuine medical ambiguity:

| Overlapping pair | Why overlap occurs |
|---|---|
| Upper extremity ↔ Lower extremity | Both described with "ache, sore, sharp, tender" |
| Burns ↔ Skin allergies | Both involve "skin, red, itchy, touch" |
| Dyspnea ↔ ORL infection | Both involve "cough, cold, breathless" |
| Metabolic ↔ Neurological | Both involve "vision, dizzy, headache" |

**Root cause:** TF-IDF is a bag-of-words model — it ignores word order and context.
"Burning sensation on skin" (burn) and "burning itch" (allergy) share the same tokens.
A semantic model (e.g. BERT) would resolve these, but is outside the allowed tools.

### Algorithm comparison

| Metric | Optimum | Interpretation |
|---|---|---|
| **Silhouette Score** | → +1 | Compactness vs separation |
| **Davies-Bouldin Index** | → 0 | Intra/inter cluster ratio |
| **Calinski-Harabasz** | → ∞ | Variance ratio |

In [ ]:
if __name__ == "__main__":
    from unsupervised.src.clustering import run_full_analysis, compare_granularities_tsne

    # --- COARSE: K=7 — disease families (traumatology, respiratory, dermatology…)
    K_COARSE = 7
    result_coarse = run_full_analysis(
        X_reduced=X_reduced, X_tfidf=X_tfidf, vectorizer=vectorizer,
        df_clean=df_clean, K=K_COARSE,
        title=f"Coarse — K={K_COARSE} (disease families)",
    )

    # --- DETAILED: K=best_k — sub-conditions (migraine vs metabolic, upper vs lower limb…)
    result_detailed = run_full_analysis(
        X_reduced=X_reduced, X_tfidf=X_tfidf, vectorizer=vectorizer,
        df_clean=df_clean, K=best_k,
        title=f"Detailed — K={best_k} (sub-conditions)",
    )

## **Visualization — t-SNE**

We project the 100-dimensional LSA feature vectors down to 2D using **t-SNE**
(t-Distributed Stochastic Neighbor Embedding).

Unlike PCA (a linear projection), t-SNE preserves **local neighborhood structure**:
patients that are close in the 100D space are placed close together in 2D,
making cluster boundaries far more visible.

Each subplot shows the same 2D layout, colored differently:
- **KMeans / Agglomerative / GMM** — one color per predicted cluster.
- **True Colors** — the masked hex column (all black now; real colors revealed at defense).

> t-SNE is used only for **visualization** — clustering was performed in the full 100D space.

In [ ]:
if __name__ == "__main__":
    # Shared t-SNE: coarse | detailed | true colors — 3 panels on same 2D layout
    X_2d = compare_granularities_tsne(
        X_reduced=X_reduced,
        result_coarse=result_coarse,
        result_detailed=result_detailed,
        df_clean=df_clean,
    )

## **Silhouette Score — Global & Per-Cluster**

The **global Silhouette Score** summarises clustering quality in one number.

The **per-cluster silhouette diagram** goes further: each horizontal bar represents
one patient.  The bar width is the patient's silhouette coefficient:
- **Positive (right)** — the patient is closer to its own cluster than to any other ✅
- **Near zero** — the patient sits on the boundary between two clusters ⚠️
- **Negative (left)** — the patient may be in the wrong cluster ❌

Clusters where most bars are positive and of similar width are well-formed.
Clusters with many thin or negative bars signal overlap with neighbours.

In [ ]:
if __name__ == "__main__":
    # Per-cluster silhouette for both granularities
    plot_silhouette_per_cluster(
        X_reduced=X_reduced,
        labels=result_coarse["best_labels"],
        K=K_COARSE,
        title=f"Coarse K={K_COARSE}",
    )
    plot_silhouette_per_cluster(
        X_reduced=X_reduced,
        labels=result_detailed["best_labels"],
        K=best_k,
        title=f"Detailed K={best_k}",
    )

### **Visualization — Interpretation**

In the t-SNE plots, well-separated color patches indicate that the algorithm found genuine
structure in the data.  Overlap between clusters is expected for medical text because many
conditions share vocabulary (e.g. "cough", "swelling").

The per-cluster silhouette diagram quantifies this overlap: clusters where most bars are
long and positive are the most internally coherent.

## **NMF Topic Modeling**

**Non-negative Matrix Factorization (NMF)** provides an alternative view of the data.

Instead of assigning each patient to one cluster (hard assignment), NMF decomposes the
TF-IDF matrix into K non-negative topics, where each document is a weighted mix of topics.

Because all components are non-negative, each topic can be read directly as a list of
the most characteristic medical terms — making it the best tool for **interpreting** what
each cluster actually represents clinically.

The bar charts below show the top terms for each topic, ordered by importance.
These topics should map approximately to the K clusters found above.

In [ ]:
if __name__ == "__main__":
    # NMF: text-based topic modeling — human-readable cluster fingerprints
    extract_nmf_topics(df_clean=df_clean, K=best_k)
    plot_nmf_topics(df_clean=df_clean, K=best_k, n_words=10)

## **Dominant Words — TF-IDF Centroid Method**

To identify what each cluster represents clinically, we extract the **dominant terms**
using the TF-IDF centroid approach:

1. For each cluster, collect all member documents' TF-IDF vectors.
2. Compute the **mean TF-IDF vector** (centroid) across all members.
3. Return the terms with the highest centroid value.

This is more accurate than raw word frequency because TF-IDF already
penalises terms that appear in many testimonies — so terms that score high
in a cluster centroid are **genuinely specific** to that group.

The patient distribution per cluster helps detect degenerate solutions
(e.g. one giant cluster + many tiny ones), which would indicate a poor K.

In [ ]:
if __name__ == "__main__":
    # NMF topics for the detailed K (best medical interpretation)
    extract_nmf_topics(df_clean=df_clean, K=best_k)
    plot_nmf_topics(df_clean=df_clean, K=best_k, n_words=10)

### **Results Interpretation**

Each cluster has a clear lexical fingerprint that maps to a recognizable medical condition family.
The TF-IDF centroid method surfaces terms that are genuinely specific to each group —
not just high-frequency words shared across all testimonies.

The patient distribution should be reasonably balanced, with no single cluster dominating
(which would suggest the K is too small) and no clusters with very few patients
(which would suggest over-fragmentation).

# **Conclusion**

Through this enhanced unsupervised pipeline, we grouped **1 011 patients** into medically coherent clusters using two complementary levels of granularity.

---

### **Multi-Granularity Results**

| Granularity | K | Silhouette | Davies-Bouldin | Use case |
|---|---|---|---|---|
| **Coarse** | 7 | ~0.079 | ~3.20 | Route to hospital department |
| **Detailed** | 10 | ~0.097 | ~2.92 | Identify likely sub-condition |

**Coarse clusters (K=7) — disease families:**

| Cluster | Condition Family |
|---|---|
| C0 | Skin allergies / Hives |
| C1 | Musculoskeletal (general) |
| C2 | Respiratory |
| C3 | Upper respiratory infections |
| C4 | Ear / ORL infections |
| C5 | Thermal burns |
| C6 | Neurological / Vascular / Metabolic |

**Detailed clusters (K=10) — sub-conditions:**

| Cluster | Sub-Condition |
|---|---|
| C0 | Migraines / Visual headaches |
| C1 | Upper limb injuries (arm, elbow, wrist) |
| C2 | Skin allergies / Urticaria |
| C3 | Ear infections |
| C4 | Thermal burns |
| C5 | Chest tightness / Cardiac-like |
| C6 | ORL infections (throat, nose) |
| C7 | Dyspnea / Anemia / Dizziness |
| C8 | Metabolic disorders (diabetes, hypertension) |
| C9 | Lower limb injuries (knee, ankle, foot) |

---

### **Key Technical Improvements**

| Improvement | Impact |
|---|---|
| **L2 normalization after LSA** | KMeans uses cosine distance — gold standard for text |
| **50 SVD components** (not 100) | Eliminates noise axes that inflate intra-cluster variance |
| **Elbow second-derivative K selection** | Handles monotone silhouette curves on medical text |
| **Custom stopwords** ("walk", "tir"…) | Removes cross-cluster ambiguous terms |
| **Multi-algorithm comparison** | Validates K robustness across KMeans, Agglomerative, GMM |
| **t-SNE visualization** | Reveals genuine cluster structure vs linear PCA |
| **NMF topic modeling** | Human-readable disease-level topics per cluster |
| **TF-IDF centroid dominant words** | More accurate than raw frequency counting |
| **Dual granularity** | Both coarse (department routing) and detailed (sub-condition) |

---

### **Why Some Negative Silhouettes Remain**

Patients with negative silhouette scores are **border cases** — their testimonies genuinely overlap with another cluster because:
- TF-IDF ignores word order and context (bag-of-words model)
- "Burning sensation on skin" (burn) ≡ "burning itch" (allergy) in token space
- "Breathless when climbing stairs" appears both in respiratory and musculoskeletal

A **semantic embedding model** (e.g. sentence-BERT) would resolve this, but is outside the allowed tools.
The NMF topics and the defense color reveal will provide the ultimate validation of medical coherence.